# Unduh full-scene SNI dari Roboflow ke Google Drive

Notebook ini hanya mengunduh dan mengarsipkan dua dataset sumber asli. **Tidak ada crop, sintesis, atau training.**

Sebelum menjalankan, buat Colab Secret dengan nama tepat `ROBOFLOW_API_KEY` dan aktifkan akses notebook. Jangan menulis API key di cell.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import json, os, shutil, subprocess, sys, tarfile

drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'roboflow'], check=True)

RAW_ROOT = Path('/content/sni-detection-fullscene-raw')
DRIVE_ROOT = Path('/content/drive/MyDrive/coffee-sni-detection-fullscene-v1')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

SOURCES = [
    {
        'name': 'adrian_detection',
        'workspace': 'situju-kamkape',
        'project': 'yolo-skripsi-2-lh14g-y61eh',
        'version': 1,
        'format': 'coco',
    },
    {
        'name': 'faruq_segmentation',
        'workspace': 'situju-kamkape',
        'project': 'robusta_sni_dataset-hr9ci',
        'version': 1,
        'format': 'coco-segmentation',
    },
]

print('RAW LOKAL :', RAW_ROOT)
print('ARSIP DRIVE:', DRIVE_ROOT)
print('TRAINING   : TIDAK DIJALANKAN')

In [ ]:
# Unduh hanya sumber yang arsip finalnya belum tersedia.
from roboflow import Roboflow

try:
    api_key = userdata.get('ROBOFLOW_API_KEY')
except Exception as error:
    raise RuntimeError(
        "Colab Secret 'ROBOFLOW_API_KEY' belum tersedia atau belum diberi akses."
    ) from error
if not api_key:
    raise RuntimeError("Colab Secret 'ROBOFLOW_API_KEY' kosong.")

rf = Roboflow(api_key=api_key)
download_roots = {}
for source in SOURCES:
    name = source['name']
    archive = DRIVE_ROOT / f'{name}.tar'
    destination = RAW_ROOT / name
    if archive.is_file():
        print('SKIP DOWNLOAD; arsip sudah ada:', archive, flush=True)
        download_roots[name] = None
        continue

    annotations = list(destination.rglob('_annotations.coco.json')) if destination.is_dir() else []
    if len(annotations) != 3:
        if destination.exists():
            shutil.rmtree(destination)
        destination.parent.mkdir(parents=True, exist_ok=True)
        print(f"DOWNLOAD {name}: {source['project']} v{source['version']} ({source['format']})", flush=True)
        dataset = (
            rf.workspace(source['workspace'])
              .project(source['project'])
              .version(source['version'])
              .download(source['format'], location=str(destination))
        )
        destination = Path(dataset.location).expanduser().resolve()
        print('LOKASI SDK:', destination, flush=True)

    annotations = sorted(destination.rglob('_annotations.coco.json'))
    images = sorted(
        path for path in destination.rglob('*')
        if path.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
    )
    if len(annotations) != 3 or not images:
        raise RuntimeError(
            f'Download {name} tidak lengkap: annotations={len(annotations)}, images={len(images)}'
        )
    print(f'SIAP ARSIP {name}: {len(images)} gambar, {len(annotations)} anotasi', flush=True)
    download_roots[name] = destination

api_key = None
del rf
print('DOWNLOAD SELESAI; API key tidak disimpan ke file.')

In [ ]:
# Simpan sebagai satu TAR per dataset agar Google Drive tidak menangani ribuan file kecil.
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
manifest = {
    'format': 'coffee_detector.sni_fullscene_archives.v1',
    'contains_original_full_scenes': True,
    'training_executed': False,
    'sources': {},
}

for source in SOURCES:
    name = source['name']
    archive = DRIVE_ROOT / f'{name}.tar'
    local_root = download_roots.get(name)
    if not archive.is_file():
        if local_root is None or not local_root.is_dir():
            raise FileNotFoundError(f'Folder lokal untuk {name} tidak tersedia.')
        files = sorted(path for path in local_root.rglob('*') if path.is_file())
        partial = Path(str(archive) + '.part')
        if partial.exists():
            partial.unlink()
        print(f'ARSIP {name}: {len(files)} file -> {archive.name}', flush=True)
        with tarfile.open(partial, 'w') as handle:
            for index, path in enumerate(files, 1):
                relative = path.relative_to(local_root).as_posix()
                handle.add(path, arcname=f'{name}/{relative}', recursive=False)
                if index % 500 == 0 or index == len(files):
                    print(f'  {name}: {index}/{len(files)} file', flush=True)
        partial.replace(archive)
        print('SAVED:', archive, flush=True)
    else:
        print('SKIP ARSIP; sudah ada:', archive, flush=True)

    with tarfile.open(archive, 'r') as handle:
        members = [member for member in handle.getmembers() if member.isfile()]
    image_count = sum(Path(member.name).suffix.lower() in IMAGE_SUFFIXES for member in members)
    annotation_count = sum(Path(member.name).name == '_annotations.coco.json' for member in members)
    if image_count <= 0 or annotation_count != 3:
        raise RuntimeError(
            f'Arsip {name} tidak valid: images={image_count}, annotations={annotation_count}'
        )
    manifest['sources'][name] = {
        **source,
        'archive': str(archive),
        'bytes': archive.stat().st_size,
        'members': len(members),
        'images': image_count,
        'annotation_files': annotation_count,
    }

manifest_path = DRIVE_ROOT / 'complete.json'
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False))
print('SAVED:', manifest_path)
print('TRAINING TIDAK DIJALANKAN.')

In [ ]:
summary = json.loads((DRIVE_ROOT / 'complete.json').read_text())
print('=== FULL-SCENE ARCHIVE SELESAI ===')
for name, row in summary['sources'].items():
    print(f"{name:20s}: images={row['images']}, annotations={row['annotation_files']}, size={row['bytes']/1e9:.2f} GB")
print('Folder:', DRIVE_ROOT)
print('Berikutnya: audit dan canonical grouped split 21 kelas; belum training.')